# Module 13: Recommendation Optimization & Latency Tuning
## Threshold Grid Search, Precision-Recall Trade-offs & Caching

This notebook demonstrates:
1. Loading the `ThresholdTuner` grid search results.
2. Plotting the Precision, Recall, F1, and Rejection Rate trade-off curves.
3. Identifying the mathematically optimal cutoff threshold $\tau^*$.
4. Profiling latency distributions and throughput for fast outfit assembly.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import REPORTS_DIR
from src.optimizer import ThresholdTuner, OptimizedOutfitRecommender

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)

### 1. Load Precomputed Optimization Results

In [ ]:
opt_file = REPORTS_DIR / "optimization_results.json"
with open(opt_file, "r", encoding="utf-8") as f:
    results = json.load(f)

tt = results["threshold_tuning"]
print(f"Optimal Threshold (tau*) : {tt['optimal_threshold']:.3f}")
print(f"Maximum F1-Score         : {tt['max_f1_score']:.4f}")
print(f"Precision at tau*        : {tt['precision_at_optimal']:.4f}")
print(f"Recall at tau*           : {tt['recall_at_optimal']:.4f}")

lat = results["latency_optimization"]
print(f"\nMean Assembly Latency    : {lat['mean_latency_ms']:.2f} ms")
print(f"p95 Assembly Latency     : {lat['p95_latency_ms']:.2f} ms")
print(f"Throughput               : {lat['throughput_outfits_per_sec']} outfits/sec")

### 2. Plotting Threshold Tuning Curves (Precision, Recall & F1-Score)

In [ ]:
curve_df = pd.DataFrame(tt["grid_curve"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Precision, Recall, F1 Curve
axes[0].plot(curve_df["threshold"], curve_df["precision"], label="Precision", color="darkgreen", linewidth=2.5)
axes[0].plot(curve_df["threshold"], curve_df["recall"], label="Recall", color="royalblue", linewidth=2.5)
axes[0].plot(curve_df["threshold"], curve_df["f1_score"], label="F1-Score", color="crimson", linewidth=3, linestyle="--")
axes[0].axvline(tt["optimal_threshold"], color="black", linestyle=":", label=f"Optimal tau*={tt['optimal_threshold']:.2f}")
axes[0].set_title("Precision-Recall-F1 vs Cutoff Threshold", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Compatibility Threshold (tau)")
axes[0].set_ylabel("Metric Score")
axes[0].legend(loc="lower left")

# Rejection Rate Curve
axes[1].plot(curve_df["threshold"], curve_df["rejection_rate"] * 100, color="purple", linewidth=2.5, marker="o")
axes[1].axvline(tt["optimal_threshold"], color="black", linestyle=":", label=f"Optimal tau*={tt['optimal_threshold']:.2f}")
axes[1].set_title("Candidate Rejection Rate (%)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Compatibility Threshold (tau)")
axes[1].set_ylabel("Rejection Rate (%)")
axes[1].legend()

plt.tight_layout()
plt.show()

### 3. Fast Outfit Generation Demonstration

In [ ]:
opt_engine = OptimizedOutfitRecommender(compatibility_threshold=tt["optimal_threshold"])
seed_id = opt_engine.manager.index_df.iloc[0]["id"]

outfits, latency_ms = opt_engine.generate_outfit_fast(seed_id, top_k=2)
print(f"Generated {len(outfits)} full outfits in {latency_ms:.2f} ms!")
print(f"Cohesion Score: {outfits[0]['cohesion_score']:.3f}")